# Corrected Court of Appeal Judgment Outcome Classification

This notebook is a **clean, runnable version** for your current objective:

**Goal:** classify the final **Court of Appeal judgment outcome** from case text and related structured features.

## What is corrected here
- **Judgment-classification framing** is explicit. This notebook is **not** for pre-judgment prediction.
- **Split happens before fitting** any TF-IDF or model-selection step.
- **No SMOTE / oversampling before cross-validation.**
- **Group-aware split** is used so the same `court_of_appeal_case_no` does not leak across train and test.
- **Temporal ordering** is respected using `judgment_date_coa` year at the group level.
- Final evaluation is done on **one untouched test set**.

> Because this notebook uses `court_of_appeal_analysis_summary`, it should be described as **judgment outcome classification**, not pre-decision prediction.


In [5]:
# !pip uninstall -y numpy pandas
# !pip cache purge
# !pip install --upgrade pip setuptools wheel
# !pip install "numpy<2" "pandas>=2.2,<3" scikit-learn imbalanced-learn matplotlib jupyter ipykernel
# %pip install --upgrade pip
# %pip install pandas numpy scikit-learn matplotlib openpyxl joblib
# %pip uninstall -y pandas
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

ImportError: cannot import name 'VisibleDeprecationWarning' from 'numpy' (unknown location)

In [9]:
import pandas as pd
# ---------- Paths ----------
NOTEBOOK_DIR = Path.cwd()
DATA_CANDIDATES = [
    NOTEBOOK_DIR / "merged_conversions.csv",
    Path("/mnt/data/merged_conversions.csv"),
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("merged_conversions.csv not found in current folder or /mnt/data")

OUTPUT_DIR = NOTEBOOK_DIR / "corrected_judgment_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Outputs:", OUTPUT_DIR)

Dataset: C:\Users\shant\Research\merged_conversions.csv
Outputs: C:\Users\shant\Research\corrected_judgment_outputs


In [10]:
# ---------- Load data ----------
df = pd.read_csv(DATA_PATH)

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].replace({r"^\s*$": np.nan}, regex=True)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
df.head(2)

AttributeError: module 'pandas' has no attribute 'read_csv'

In [ ]:
# ---------- Target cleaning ----------
def normalize_outcome(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if "partly" in text:
        return "Partly Allowed"
    if "allow" in text:
        return "Appeal Allowed"
    if "dismiss" in text:
        return "Appeal Dismissed"
    return np.nan

work = df.copy()
work["target"] = work["coa_final_outcome_class"].map(normalize_outcome)
work = work[work["target"].notna()].copy()

work["judgment_date_coa_dt"] = pd.to_datetime(work["judgment_date_coa"], errors="coerce")
median_year = int(work["judgment_date_coa_dt"].dt.year.dropna().median())
work["coa_year"] = work["judgment_date_coa_dt"].dt.year.fillna(median_year).astype(int)

# Use case number as group; if missing, fall back to conversion_id
work["group_id"] = work["court_of_appeal_case_no"].astype("string")
missing_group = work["group_id"].isna() | (work["group_id"].str.strip() == "")
work.loc[missing_group, "group_id"] = work.loc[missing_group, "conversion_id"].astype("string")

print("Labeled rows:", len(work))
print("Unique groups:", work["group_id"].nunique())
print(work["target"].value_counts())

In [ ]:
# ---------- Build text + structured feature columns ----------
TEXT_COLS = [
    "brief_facts_summary",
    "grounds_of_appeal_raw_text_summary",
    "court_of_appeal_analysis_summary",
]

work["combined_text"] = (
    work[TEXT_COLS]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

FLAG_COLS = [c for c in work.columns if c.startswith("gnd_") and c != "gnd_other_description"]
FLAG_COLS += [
    "eyewitness_present",
    "child_witness_present",
    "expert_evidence_present",
    "forensic_evidence_present",
    "dying_declaration_present",
    "confession_present",
    "circumstantial_case",
    "legal_errors_identified",
    "procedural_defects_present",
    "directions_on_burden_of_proof_correct",
    "digital_evidence_present",
    "weapon_recovered",
    "motive_established",
    "failure_to_cross_examine_material_points",
    "hospital_treatment_details_present",
    "hc_errors_identified_by_coa",
    "order_on_retrial",
    "release_ordered",
]
FLAG_COLS = [c for c in FLAG_COLS if c in work.columns]

CAT_COLS = [
    "offence_category",
    "high_court_location",
    "appeal_type",
    "hc_sentence_type",
    "plea_of_accused",
    "trial_method",
] + FLAG_COLS
CAT_COLS = [c for c in CAT_COLS if c in work.columns]

NUM_COLS = [c for c in ["num_prosecution_witnesses", "num_defence_witnesses", "coa_year"] if c in work.columns]

for c in NUM_COLS:
    work[c] = pd.to_numeric(work[c], errors="coerce")

MODEL_COLS = ["combined_text"] + CAT_COLS + NUM_COLS + ["target", "group_id", "coa_year"]
model_df = work[MODEL_COLS].copy()

print("Text column ready:", model_df['combined_text'].notna().mean())
print("Categorical columns:", len(CAT_COLS))
print("Numeric columns:", NUM_COLS)
model_df.head(2)

In [ ]:
# ---------- Group-aware temporal split ----------
# Sort unique groups by median CoA year and keep the latest 20% groups as test.
group_year = (
    model_df.groupby("group_id", dropna=False)["coa_year"]
    .median()
    .sort_values(kind="stable")
)

cut_index = max(1, int(len(group_year) * 0.80))
train_groups = set(group_year.index[:cut_index])
test_groups = set(group_year.index[cut_index:])

train_mask = model_df["group_id"].isin(train_groups)
test_mask = model_df["group_id"].isin(test_groups)

X_train = model_df.loc[train_mask, ["combined_text"] + CAT_COLS + NUM_COLS].copy()
X_test = model_df.loc[test_mask, ["combined_text"] + CAT_COLS + NUM_COLS].copy()
y_train = model_df.loc[train_mask, "target"].copy()
y_test = model_df.loc[test_mask, "target"].copy()
groups_train = model_df.loc[train_mask, "group_id"].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())
print("\nUnique train groups:", len(set(groups_train)))
print("Unique test groups:", len(test_groups))
print("Group overlap:", len(set(groups_train).intersection(test_groups)))

In [ ]:
# ---------- Preprocessor ----------
preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                max_features=2500,
                min_df=3,
                ngram_range=(1, 2),
                sublinear_tf=True,
                strip_accents="unicode"
            ),
            "combined_text",
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                ]
            ),
            CAT_COLS,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler(with_mean=False)),
                ]
            ),
            NUM_COLS,
        ),
    ],
    remainder="drop",
)

In [ ]:
# ---------- Candidate models + train-only CV ----------
# No oversampling here. We use class_weight='balanced' and do model selection only on training data.
cv = GroupKFold(n_splits=5)

candidates = {
    "LogisticRegression": (
        LogisticRegression(
            max_iter=2500,
            solver="saga",
            class_weight="balanced",
            multi_class="auto",
        ),
        {
            "model__C": [0.5, 1.0, 2.0],
        },
    ),
    "LinearSVC": (
        LinearSVC(
            class_weight="balanced",
            dual="auto",
        ),
        {
            "model__C": [0.5, 1.0, 2.0],
        },
    ),
}

results = []
best_estimators = {}

for name, (estimator, param_grid) in candidates.items():
    pipe = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", estimator),
        ]
    )

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )

    search.fit(X_train, y_train, groups=groups_train)
    best_estimators[name] = search.best_estimator_
    results.append(
        {
            "model": name,
            "best_params": search.best_params_,
            "cv_macro_f1": search.best_score_,
        }
    )

cv_results = pd.DataFrame(results).sort_values("cv_macro_f1", ascending=False).reset_index(drop=True)
cv_results

In [ ]:
# ---------- Fit best model on full train and evaluate once on untouched test ----------
best_model_name = cv_results.loc[0, "model"]
best_pipeline = best_estimators[best_model_name]

y_pred = best_pipeline.predict(X_test)

metrics = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_macro_f1": f1_score(y_test, y_pred, average="macro"),
            "test_weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        }
    ]
)

print("Best model from train-only CV:", best_model_name)
display(metrics)

print("\nClassification report:")
print(classification_report(y_test, y_pred))

In [ ]:
# ---------- Confusion matrix ----------
labels = ["Appeal Allowed", "Appeal Dismissed", "Partly Allowed"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm)

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix - {best_model_name}")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Optional: inspect strongest linear features ----------
# Works for both LogisticRegression and LinearSVC because both are linear here.
feature_names = best_pipeline.named_steps["prep"].get_feature_names_out()
model = best_pipeline.named_steps["model"]

coef = model.coef_
classes = list(model.classes_)

top_features = []
for class_idx, class_name in enumerate(classes):
    top_idx = np.argsort(coef[class_idx])[-15:][::-1]
    for rank, idx in enumerate(top_idx, start=1):
        top_features.append(
            {
                "class": class_name,
                "rank": rank,
                "feature": feature_names[idx],
                "weight": coef[class_idx, idx],
            }
        )

top_features_df = pd.DataFrame(top_features)
top_features_df.head(30)

In [ ]:
# ---------- Save artifacts ----------
joblib.dump(best_pipeline, OUTPUT_DIR / "judgment_outcome_pipeline.joblib")

metadata = {
    "task": "Court of Appeal judgment outcome classification",
    "framing_note": "Uses court_of_appeal_analysis_summary; this is judgment classification, not pre-judgment prediction.",
    "data_path": str(DATA_PATH),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_groups": int(len(set(groups_train))),
    "test_groups": int(len(test_groups)),
    "best_model": best_model_name,
    "cv_results": cv_results.to_dict(orient="records"),
    "test_metrics": metrics.to_dict(orient="records"),
    "labels": sorted(y_train.unique().tolist()),
    "text_columns": TEXT_COLS,
    "categorical_columns": CAT_COLS,
    "numeric_columns": NUM_COLS,
}

with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

cv_results.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
top_features_df.to_csv(OUTPUT_DIR / "top_linear_features.csv", index=False)

print("Saved:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

In [ ]:
# ---------- Simple prediction helper ----------
loaded_pipeline = joblib.load(OUTPUT_DIR / "judgment_outcome_pipeline.joblib")

default_values = {}
for c in CAT_COLS:
    mode = model_df[c].mode(dropna=True)
    default_values[c] = mode.iloc[0] if len(mode) else "Unknown"

for c in NUM_COLS:
    default_values[c] = float(model_df[c].median()) if model_df[c].notna().any() else 0.0

def predict_judgment_outcome(
    brief_facts_summary="",
    grounds_of_appeal_raw_text_summary="",
    court_of_appeal_analysis_summary="",
    **structured_overrides
):
    row = default_values.copy()
    row["combined_text"] = " ".join(
        [
            str(brief_facts_summary or ""),
            str(grounds_of_appeal_raw_text_summary or ""),
            str(court_of_appeal_analysis_summary or ""),
        ]
    ).strip()

    for key, value in structured_overrides.items():
        if key in row:
            row[key] = value

    row_df = pd.DataFrame([row], columns=["combined_text"] + CAT_COLS + NUM_COLS)
    pred = loaded_pipeline.predict(row_df)[0]

    result = {"predicted_outcome": pred}
    if hasattr(loaded_pipeline.named_steps["model"], "decision_function"):
        scores = loaded_pipeline.decision_function(row_df)
        if np.ndim(scores) == 2:
            result["decision_scores"] = dict(zip(loaded_pipeline.named_steps["model"].classes_, scores[0].tolist()))
    return result

demo_result = predict_judgment_outcome(
    brief_facts_summary="The accused was convicted by the High Court. The appeal challenges the identification evidence and contradictions in witness testimony.",
    grounds_of_appeal_raw_text_summary="Wrong identification, contradictions, and procedural unfairness were raised.",
    court_of_appeal_analysis_summary="The court found serious weaknesses in identification and material contradictions affecting the conviction."
)
demo_result

## How to report this notebook correctly

Use wording like this:

> **This notebook classifies the final Court of Appeal judgment outcome from judgment-related text and structured case features.**

Do **not** describe it as:
- pre-judgment prediction
- forecasting before decision
- predicting future Court of Appeal outcome without judgment text

This version is built for **judgment outcome classification**.
